# **Duration And Convexity: Control Risk and Develop an Investment Strategy.**
<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-Three/blob/master/colab/Colab_Calculating_Yields_To_Maturity.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-Three/master?urlpath=lab/tree/notebooks/Calculating_Yields_To_Maturity.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>
  
  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-Three/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>

  There are two important related facts that need to be accounted for by any strategy that controls the interest rate risk of a portfolio or attempts to develop an investment strategy based upon investors' beliefs about potential changes in the term structure.

**Duration and Convexity:** These metrics are not fixed but change with changes in interest rates. Both measures are derivatives of the present value formula, and those derivatives are not constant. The last time you took a flight, you took off, flew at roughly a constant altitude, and landed. The speed, angle of approach, and altitude varied in each of those stages, and the pilots adjusted their actions according to the stage of the flight. Portfolio managers do the exact same thing when reacting to changes in interest rates.

**Non-Parallel Yield Curve Shifts:** As demonstrated in Chapter Three, changes in yields alter the shape of the term structure. For example, the inverted case of Chapter Three caused the yield to maturity of a one-year bond to increase by 82 basis points, while the twenty-nine-plus year bond decreased by 122 basis points. Understanding the state of the economy, and therefore the expected shape of the term structure, is crucial for managing interest rate risk

:::{important} [ ▼ ] How to use this page: Run, Copy, & Download
:class: dropdown

<ul>
  <li><b>⏻ Run code right here:</b> Click the <b>Power Button</b> icon at the top of the screen to activate <b>Live Code</b>.</li>
  <li><b>📋 Copy code:</b> Hover over any code block and click the <b>Clipboard icon</b> in the top-right corner.</li>
  <li><b>📥 Download this file:</b> Click the <b>Download icon</b> (downward arrow) at the top right of the screen to save this exact notebook to your computer.</li>
</ul>
:::

### Importing libraries, modules, And functions

Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported.  (see [Control Statements](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/Control_Statements.html#the-try-and-except)).

```
# Import OS to interact with local computer operating system
import os
import sys
import requests
from datetime import datetime, date
from types import ModuleType

# Import the NumPy library for numerical operations, commonly aliased as np.
try:
    import numpy as np
except:
    !pip install-q numpy
    import numpy as np

# Import the pandas library for data manipulation and analysis, aliased as pd.
try:
    import pandas as pd
except:
    !pip install -q pandas
    import pandas as pd

# Import the scipy library to find roorts of present value function
try:
  import scipy.optimize as optimize
except:
  !pip install -q scipy
  import scipy.optimize as optimize

# Import Altair library to visualize spot rates and yields to maturity
try:
  import altair as alt
except:
  !pip install -q altair
  import altair as alt
```

In [1]:
# Import OS to interact with local computer operating system
import os
import sys
import requests
from datetime import datetime, date
from types import ModuleType

# Import the NumPy library for numerical operations, commonly aliased as np.
try:
    import numpy as np
except:
    !pip install-q numpy
    import numpy as np

# Import the pandas library for data manipulation and analysis, aliased as pd.
try:
    import pandas as pd
except:
    !pip install -q pandas
    import pandas as pd

# Import the scipy library to find roorts of present value function
try:
  import scipy.optimize as optimize
  from scipy.stats import norm, beta

except:
  !pip install -q scipy
  import scipy.optimize as optimize
  from scipy.stats import norm, beta

# Import Altair library to visualize spot rates and yields to maturity
try:
  import altair as alt
except:
  !pip install -q altair
  import altair as alt

:::{important} ☁️ Cloud-Loading: How In-Memory Modules Work
:class: dropdown

**The Logic:**
Usually, Python looks for modules as `.py` files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.

**The Workflow:**
1. **Fetch:** `requests.get(url)` grabs the raw text of your Python script from Dropbox.
2. **Instantiate:** `ModuleType(module_name)` creates an empty "container" in your computer's RAM.
3. **Execute:** `exec(code, module.__dict__)` runs that text inside the container, turning text into live functions.
4. **Register:** By adding it to `sys.modules`, we tell Python: *"If I try to import this later, don't look on the disk—look right here in the memory."*

**Why do this?**
It makes your notebooks **100% portable**. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."
:::

In [2]:
## Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.content.decode('utf-8'), module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import(create_payoff_df,
                                            ns_spot_rates,
                                            calc_ytm)


    print(f"✅ Successfully imported module from URL.")
    # Assign the FredReader attribute
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")

✅ Successfully imported module from URL.


## 🧠**Simulation of the Term Structure that Incorporate Beliefs**

Portfolio managers don't live in a vacuum. They are required to have a strong knowledge of the current state of the economy and financial markets and how risks might be expressed.  The examples we rely on here are FED policies.  Two opposite strategies are considered with unsurprisingly conflicting portfolio management implications.  Speed is good when the plane is cruising but not so when it's landing.

### 🎲**Setting Bounds and Loading the Dice**

The first place a portfolio manager has to come to is what he thinks is possible.  If it's a cut in the short rate, how big or small could it be? Beliefs about quantitative tightening or easing have to be expressed in some tractable way.  Like the pilot adjusting to the state of the flight, the portfolio manager needs to understand what's likely to happen next.

To put beliefs into practice, probabilities have to be assigned to the ranges of values.  For that the portfolio managers need an intuitive way to convert beliefs into a mathematical relation-not something that is written in stone, but provides a method of quantifying and understanding what matters and what doesn't.  The pilot needs instruments, the portfolio needs a recipe.

These needs can be met with the Beta distribution.  It's nothing like the beta that you hear about everyday, but a very convenient and intuitive way to express beliefs in a quantitative way.

The Beta distribution lets us control both the minimum and maximum values and how the probability is allocated between those values.  There are two parameters that control the allocation of probability: $\alpha \text{ and  } \beta$.  These parameters do two things:

* ⚖️ **Skew or Tilt of the distribution**:  
  * if $\alpha >  \beta$ the distribution is more concentrated to the right tilted towards higher values.   
  * if $\alpha <  \beta$ the distribution is more concentrated to the left tilted towards lower values.  
  * if $\alpha = \beta$ the distribution is centered and not tilted towards higher or lower values.  
* 🎯 **Tightness or Conviction About Values**:  
  * If  $\alpha = \beta =1$ you have no conviction about any of the possible values.  The probability of a value between any two values only depends upon the distance between the values and not their respective values.  It's called a uniform distribution.  
  * If  $\alpha = \beta = 2$ you have a belief that values in and around what you expect are likely, but the conviction isn't strong.  The probability of values away from  the expected are substantial. A bell shaped curve with plenty of area away from what expect.  
  * If  $\alpha = \beta = 20$ you have a strong belief that values will be close to your expectation. Large deviations are unlikely. A bell shaped curve with a small area away from what you expect.

> **Note**: Bear in mind that both the tilt and conviction are at work for any pair of  $\alpha \text{ and } \beta$.

## 🧩 **The Cohesion Problem: Preventing Impossible Outcomes**
Once a portfolio manager has customized their individual beliefs for Level, SOFR, and Curvature using Beta distributions, a major hazard emerges. If you pull random values from each of these three distributions independently, you are highly likely to create impossible market outcomes.

For instance, your simulation might pair an extremely high overall interest rate Level with a massive, steeply upward-sloping curve. Perhaps a useful novel thought experiment, but completely useless as a simulation to inform an actual investment strategy. In the real world, macro forces lock these variables together; when overall rate levels spike, the yield curve historically tends to flatten or invert.

If our simulation doesn't respect these fundamental market relationships, our portfolio stress tests will generate meaningless data. The pilot cannot adjust the plane's altitude without affecting its airspeed and drag; a portfolio manager cannot shift the level of the yield curve without affecting its slope.

To prevent these impossible outcomes, we need a mathematical tool that acts like connecting tissue, binding our variables together without crushing the unique individual shapes (the skews and tightness) representing our beliefs.

That tool is the *Copula*.

### 🧬 Completing the Uncertainty: Tying the Shapes Together

The word copula comes from the Latin word for "link" or "tie" (the same root as the English word couple). In quantitative finance, a copula does something incredibly elegant: it splits a market scenario's DNA into two completely independent parts:

1. **The Marginal Beta Distributions (Beliefs)**: 🧠 The individual, subjective beliefs about the boundaries, tilt, and tightness for Level, SOFR, and Curvature.

2. **The Dependence Structure (How Beliefs Relate)**: 📊 The historical, statistical reality of how these three factors move together.

#### ⚙️ How the Copula Engine Works
Instead of drawing numbers directly from the custom Beta distributions, the simulation performs a multi-step sequence behind the scenes:

1. **Draw Correlated Data**: 🎲 The engine first draws random numbers from a standard joint framework (a Multivariate Normal distribution) that is strictly governed by the market's historical correlation matrix.

2. **Extract Pure Probability**:💧It converts those raw draws into pure, correlated probabilities between 0 and 1. If the model draws a high probability for a rising Level, the correlation matrix forces it to simultaneously draw a matching probability for a high SOFR rate (a flattening slope).

3. **Map to Your Beliefs**: 🗺️ Finally, it passes those correlated probabilities into your custom Beta distributions anchoring beliefs to market realities.

The beauty of this architecture is that the Copula perfectly preserves the real-world correlation structure between Level, SOFR, and Curvature, yet allowing the final values to morph into the exact left-skewed, right-skewed, or ultra-tight shapes specified by the beliefs. It guarantees that every one of the simulations is grounded in reality. An authentic experiment in how a portfolio’s duration and convexity truly behaves.

## 📈 Going From the Correlation of Level,Slope, and Curvature to the SOFR Covariance Matrix
To execute a realistic simulation of the term structure using the Nelson-Siegel framework, we must establish how the underlying factors relate to one another. We begin with the foundational empirical estimates from the landmark paper by Diebold and Li, which find a historical correlation between the Level ($\beta_0$) and Slope ($\beta_1$) factors of approximately $-0.4$.

To isolate the impact of policy changes, we treat the short rate $r(0)$—which asymptotically acts as our SOFR proxy—as our primary innovation. Under the Nelson-Siegel framework, the short rate is explicitly defined by the structural identity:
<br>

$$r(0) = \beta_0 + \beta_1$$

<br>

Assuming normalized baseline variances for our Level and Slope parameters such that $\sigma_{\beta_0}^{2} = 1$ and $\sigma_{\beta_1}^{2} = 1$, we can use standard linear combinations of random variables to derive the implied variance of our short rate innovation and its correlation with the overall curve Level.

### 1. 📐 The Variance of the Short Rate ($r(0)$)

The variance of the sum of two correlated variables is governed by their individual variances and their covariance:
<br>

$$\sigma_{r(0)}^{2} = \sigma_{\beta_0}^{2} + \sigma_{\beta_1}^{2} + 2\rho_{\beta_0,\beta_1}\sigma_{\beta_0}\sigma_{\beta_1}$$

<br>
Plugging in our baseline values and the Diebold-Li reported estimate of $-0.4$:
<br>

$$\sigma_{r(0)}^{2} = 1 + 1 + 2(-0.4)(1)(1) = 1.2$$

$$\sigma_{r(0)} = \sqrt{1.2} \approx 1.09545$$

<br>

### 2. 🧮 The Covariance of the Level and the Short Rate
Next, we calculate how our short rate innovation co-moves with the overall Level ($\beta_0$):
<br>

$$\sigma_{r(0),\beta_0} = \text{Cov}(\beta_0 + \beta_1, \beta_0) = \sigma_{\beta_0}^{2} + \rho_{\beta_0,\beta_1}\sigma_{\beta_0}\sigma_{\beta_1}$$

$$\sigma_{r(0),\beta_0} = 1 + (-0.4)(1)(1) = 0.6$$

### 3.⚖️ The Correlation of the Level and Short Rate
The covariance of the Level and SOFR is scaled by the product of the two standard deviations:

$$\rho_{r(0),\beta_0} = \frac{\sigma_{r(0),\beta_0}}{\sigma_{r(0)}\sigma_{\beta_0}} = \frac{0.6}{1.09545 \times 1} \approx 0.548$$

<br>

### 4.🗂️ The Covariance Matrix of the Multivariate Normal Simulations($\Sigma$)
By substituting our derived short-rate volatility ($\sigma_{r(0)} \approx 1.09545$) and our adjusted correlation ($0.548$) alongside standard historical estimates for Curvature ($\beta_2$), we construct the finalized covariance matrix $\Sigma$ that will feed directly into np.random.multivariate_normal

<br>

$$\Sigma = \begin{bmatrix}
1.0 & 0.548 & 0.20 \\
0.548 & 1.20 & 0.329 \\
0.20 & 0.329 & 1.0
\end{bmatrix}$$
<br>

> 💡 **Crucial Scale Note:** When we use this covariance matrix to calculate probabilities, a scale must be provided so that the probability calculations account for the elevated standard deviation of SOFR (1.09 versus 1.0 for the Level ($\beta_0$) and Curvature ($\beta_2$).


## 🎲💧 Generate The Correlated Proabilities (Cumulative Densities)
First, we define our covariance matrix for the Level, SOFR, and Curvature. We will simulate a multivariate normal distribution and then convert those values into uniform probabilities using the Normal Cumulative Density Function (CDF).

In [20]:
# The uninformed covariance matrix (Level, SOFR, Curvature)
cov_matrix = np.array([
    [1.0,   0.548, 0.20],  # Level
    [0.548, 1.20,  0.329], # SOFR (Variance = 1.2)
    [0.20,  0.329, 1.0]    # Curvature
])

# Simulate the uninformed joint distribution
means = np.zeros(3)
num_sims = 10000
z_normal = np.random.multivariate_normal(means, cov_matrix, size=num_sims)

# Specific scales (standard deviations) so 1.0 is not assumed for all variables
# [sqrt(1.0), sqrt(1.2), sqrt(1.0)] equals [1.0, 1.09545, 1.0]
scales = np.sqrt(np.diag(cov_matrix))

# Generate the uniform probabilities using the exact scale vector
# NumPy automatically broadcasts 'scales' across the 3 columns of z_normal
corr_cdfs = norm.cdf(z_normal, loc=0, scale=scales)

# Verify the results for SOFR... it should go from 0 to 1
print(f"Column 1 (SOFR) Uniform Bounds: [{corr_cdfs[:, 1].min():.4f}, {corr_cdfs[:, 1].max():.4f}]")

Column 1 (SOFR) Uniform Bounds: [0.0001, 0.9995]


## ⚖️🎯 Convert the Correlated Probabiities to Correlated *Tilted* and *Convicted* Betas
Now, we take those correlated uniform probabilities ($0$ to $1$) and pass them through the inverse Cumulative Distribution Function (beta.ppf) of our custom Beta distributions. The relative values of $\alpha$ and $\beta$ determine the dynamic tilt (skew) and our specific conviction (spread) regarding current Fed actions..

### 🏛️ Scenario Setup: A Fed Cut Coincident with Quantitative Tightening (QT)

* **Level ($\beta_0$)**: TThe long-term anchor. Set as symmetric ($\alpha = \beta = 2.0$) across a normal range from 3.68% to 7.68% (centered at 5.68%).
* **SOFR (r(0))**: TThe monetary policy target. Tilted Left / Right-Skewed ($\alpha = 2.0, \beta = 6.0$), capping its range tightly between 3.0% and 3.6% to reflect a high-conviction projected Fed rate cut.
* **Curvature ($\beta_2$)**:The bond supply factor. Tilted Right / Left-Skewed ($\alpha = 6.0, \beta = 2.0$), pushed higher to reflect the massive influx of term bond supply from Quantitative Tightening.


In [21]:
# The relative values of alpha and beta determine the tilt and conviction
a_params = np.array([2.0, 2.0, 6.0])  # Shape alpha
b_params = np.array([2.0, 6.0, 2.0])  # Shape beta

# Vectorize calculation of tilted and convicted beta distributions
standard_beta_matrix = beta.ppf(corr_cdfs, a_params, b_params)

### 📊 Plotting 'The Beliefs' via the Copula Engine
With our arrays mapped, we translate the standard $[0, 1]$ beta outputs into real-world interest rate boundaries. The final step scales these outputs to their respective financial ranges, tracking exactly how our macro view reshapes our simulated distribution paths.

In [6]:
mins = np.array([0.050, 0.03, -0.0500])
maxs = np.array([0.070,  0.036,  0.0500])
level=0.05+[0.07-0.05]*standard_beta_matrix[:,0]
sofr=0.0325+[0.036-0.0325]*standard_beta_matrix[:,1]
curvature=-0.05+[0.05--0.05]*standard_beta_matrix[:,2]

df1 = pd.DataFrame({'value': level, 'distribution': 'Beta(2,2)'})
df2 = pd.DataFrame({'value': sofr, 'distribution': 'Beta(2,6)'})
df3 = pd.DataFrame({'value': curvature, 'distribution': 'Beta(6,2)'})
alt.data_transformers.disable_max_rows()

def create_chart(data, title, x_min, x_max, color):
    # Calculate the mean dynamically from the dataset
    mean_val = data['value'].mean()
    # Format the x-axis label
    x_label = f"Mean Equals {mean_val:.4f}"

    return alt.Chart(data).mark_bar(color=color).encode(
        alt.X('value:Q',
              bin=alt.Bin(maxbins=30, extent=[x_min, x_max]),
              scale=alt.Scale(domain=[x_min, x_max]),
              title=x_label),  # Use the dynamically created label
        alt.Y('count()', title='Count')
    ).properties(
        width=200,
        height=200,
        title=title
    )

chart1 = create_chart(df1, 'Level: Symmetric-Beta(2,2)',
                      x_min=0.045, x_max=0.075,
                      color='red')
chart2 = create_chart(df2, 'SOFR: Right Skewed-Beta(2,6)',
                      x_min=0.032,
                      x_max=0.036,
                      color='blue')
chart3 = create_chart(df3, 'Curvature: Left Skewed-Beta(6,2)',
                      x_min=-0.0475,
                      x_max=0.0525,
                      color='green')
simulated_results = alt.hconcat(chart1, chart2, chart3).resolve_scale(y='independent')
simulated_results.save('beta_histograms_mean_xlabel.json')
display(simulated_results)

alt.HConcatChart(...)

### Reviewing the Output Results
We have generated results that match the beliefs:
1. **Level ($\beta_0$)** shows a beautiful, balanced bell curve around the $0.0600$ baseline.
2. **SOFR** is tightly compressed, stacked hard against its lower bounds with a prominent right skew—exactly how a market behaves when pricing a floor on a Fed cutting cycle.
3. **Curvature ($\beta_2$)** displays a fierce left-hand skew, capturing the structural stress of term premium expansion driven by QT.

Because these were funneled through the normal copula, these custom shapes are still completely bound by the underlying empirical covariance matrix.

In [14]:
level.shape

(10000,)


::::{admonition} 📝 Check Your Understanding Of Simulations
:class: note

**Q1: Correlation of Informed (Beta) versus Uninformed (Multivariate Normal) simulations**
Do you expect the correlation of simulated values of $\beta_0$, SOFR, and $\beta_2$ to be the same as those used in the multivariate normal simulations?

**Q2: The derived distribution of the slope ($\beta_1$)**
Do you expect the shape of the derived distribution of the slope ($\beta_1$) to more closely resemble the level ($\beta_0$) or SOFR?

:::{dropdown} 👉 Click here to reveal the answers
**Answers**

**Q1:** Because the Beta distributions express specific beliefs, the final correlations will differ slightly from the uninformed multivariate normal inputs. You can test the results with a few lines of code:

```python
df_beta = pd.DataFrame({'Level': level, 'SOFR': sofr, 'Curvature': curvature})
display(df_beta.corr())
The information embedded in the beta distriubtions trims the uniformed correlations of the multivariate normal simulations.

**Q2:** The range of values for the Level ($\beta_0$) are much wider than SOFR, and that will cause the derived distribution of the Slope ($\beta_1$) to more closely resemble the level.  Because the Level ($\beta_0$) is symmetric and SOFR is tilted left (right-skewed), the derived distribution of the Slope ($\beta_1$) has a slight right tilt (left-skewed).  You can test the results with a few lines of code:
```python
df4= pd.DataFrame({'value': sofr-level, 'distribution': 'Derived'})
chart_4=create_chart(df4, 'Slope: Slightly Left Skewed-',
                     x_min=df4['value'].min(),
                     x_max=df4['value'].max(),
                     color='orange')
slope_chart=alt.hconcat(chart1, chart2, chart_4).resolve_scale(y='independent')
display(slope_chart)
```
:::
::::




### 🎣 The Catch of the Day: The Illusion of the Single Number

If you take only one thing away from this simulation exercise, let it be this: **never trust a single risk metric in a dynamic market.**

Textbooks train you to look at a 7-year Barbell and a 7-year Bullet and see the exact same portfolio. But as our Copula simulations just proved, the moment the yield curve does anything other than a perfectly parallel shift, that illusion violently shatters.

* **Duration** tells you what happens if the market behaves perfectly.
* **Convexity** warns you that the price-yield relationship is curved, not straight.
* **Simulation** forces you to face the reality that Level, Slope, and Curvature have a mind of their own.

Financial markets exist to transfer risk from those who don't want it to those who know how to manage it. By replacing static assumptions with information-packed Beta distributions and historical correlation matrices, you are taking your first steps across that dividing line.

You have reached the daily limit for this module. Save your notebook, clear your variables, and we will see you in the next session.